# MCP y A2A con LangGraph, LangChain y Gradio
**Cómo un agente usa herramientas (MCP) y cómo habla con otro agente (A2A) · Notebook resuelto (Colab o local)**

Seguimos con la **sucursal bancaria ficticia** de la sesión anterior. Hoy separamos el sistema en piezas que se comunican con **protocolos estándar**:

| Pieza | Qué es | Protocolo |
|---|---|---|
| 🧑‍💼 **Asesor** | Agente principal (LangChain + LangGraph + Groq). Habla con el cliente | — |
| 🗄️ **Servidor MCP "Core bancario"** | Expone **herramientas**: saldo y movimientos | **MCP** |
| 🤖 **Agente de Créditos** | **Otro agente**, con su propio LLM y sus propias herramientas. Simula cuotas | **A2A** |

> ⚠️ Todo es ficticio: no hay dinero real, ni aprobación de créditos, ni datos personales.
> 🔑 Necesitas `GROQ_API_KEY`: en tu PC, en el archivo `.env` de la raíz del repositorio; en Colab, en **Secrets** (icono 🔑). Nunca la pegues en el código.

## 🧠 La idea en una imagen
```
                    ┌──────────────────────────┐
   Cliente ───────► │   🧑‍💼 ASESOR (agente)     │
                    │   LLM decide qué usar    │
                    └──────┬────────────┬──────┘
                MCP        │            │        A2A
        "usa una herramienta"           "pídele a otro agente"
                           ▼            ▼
             ┌──────────────────┐   ┌──────────────────────────┐
             │ 🗄️ Servidor MCP   │   │ 🤖 Agente de Créditos      │
             │ consultar_saldo  │   │ su propio LLM + su propia │
             │ listar_movimient.│   │ herramienta simular_cuota │
             └──────────────────┘   └──────────────────────────┘
```

| | **MCP** · Model Context Protocol | **A2A** · Agent2Agent Protocol |
|---|---|---|
| Conecta | Un agente ↔ **herramientas y datos** | Un agente ↔ **otro agente** |
| Analogía | Un **enchufe universal**: cualquier agente puede usar cualquier herramienta MCP | Un **teléfono entre colegas**: le delegas una tarea a otro experto |
| El otro lado… | **No piensa**: ejecuta una función y devuelve el resultado | **Piensa**: tiene su propio LLM, decide cómo resolver |
| Descubrimiento | `tools/list` → lista de herramientas con su esquema | **Agent Card** en `/.well-known/agent-card.json` |
| Llamada | `tools/call` (JSON-RPC) | `SendMessage` (JSON-RPC) → `Message` o `Task` |
| Librerías hoy | `mcp` (servidor) + `langchain-mcp-adapters` (cliente) | `a2a-sdk` (servidor y cliente) |

**Regla corta:** *MCP para **hacer cosas**, A2A para **pedirle a alguien que haga cosas**.* Se complementan: un agente A2A puede usar MCP por dentro.

### 🧭 Recorrido
| Paso | Qué haces |
|---|---|
| 0 | Instalar y conectar con Groq |
| 1 | Bitácora compartida (para ver las interacciones) |
| **Parte A · MCP** | |
| 2 | Crear el **servidor MCP** con dos herramientas |
| 3 | Arrancarlo en segundo plano |
| 4 | Hablarle **"a mano"** con el protocolo: `initialize`, `tools/list`, `tools/call` |
| 5 | Convertir sus herramientas en herramientas de **LangChain** y dárselas a un agente |
| **Parte B · A2A** | |
| 6 | Crear el **Agente de Créditos** (LangGraph) |
| 7 | Publicarlo como **servidor A2A**: Agent Card + ejecutor |
| 8 | Descubrirlo (Agent Card) y enviarle un mensaje **"a mano"** |
| 9 | Envolver la llamada A2A como **herramienta** del Asesor |
| **Parte C · Todo junto** | |
| 10 | El **Asesor** con herramientas MCP + agente A2A |
| 11 | **Gradio**: diagrama de secuencia en vivo |


## 📘 Antes de empezar: cómo leer este notebook si no programas
No necesitas entender cada línea. En cada paso hay un recuadro **🧩 En palabras simples** con lo que hace la celda, una analogía y lo que vas a ver. Dentro del código, las líneas que empiezan con `#` son **comentarios**: explicaciones para personas; Python las ignora.

| Palabra que verás | Qué significa |
|---|---|
| **variable** (`MODELO = ...`) | Una caja con nombre donde guardamos un dato |
| **función** (`def algo(...):`) | Una receta con nombre: recibe ingredientes y devuelve un resultado |
| **diccionario** (`{'saldo': 5000000}`) | Una ficha con campos y valores, como una fila de Excel |
| **lista** (`[a, b, c]`) | Varios elementos en orden |
| **`import`** | Traer herramientas que otras personas ya programaron |
| **decorador** (`@algo` encima de una función) | Una etiqueta que le da un "superpoder" a la función, por ejemplo convertirla en herramienta MCP |
| **`async` / `await`** | "Espera a que esto termine sin bloquear lo demás". Se usa cuando hablamos por red |
| **servidor** | Un programa que se queda escuchando pedidos, como una ventanilla abierta |
| **puerto** (`8001`, `9999`) | El número de la ventanilla donde escucha cada servidor |
| **JSON** | El formato de texto con el que los programas se mandan datos (parece un diccionario) |
| **LLM** | El modelo de lenguaje (en este caso de Groq) que "piensa" y decide |
| **agente** | Un LLM con instrucciones y herramientas, que decide qué usar para responder |


## Paso 0 · ▶️ Instalar y conectar con Groq
| Paquete | Para qué |
|---|---|
| `langchain`, `langgraph` | Crear agentes (LangChain arma un grafo de LangGraph por dentro) |
| `langchain-groq` | Usar Groq como LLM en LangChain |
| `mcp` | Crear el **servidor MCP** y hablarle con el protocolo |
| `langchain-mcp-adapters` | Convertir herramientas MCP en herramientas de LangChain |
| `a2a-sdk[http-server]` | Crear el **servidor A2A** y su **cliente** |
| `python-dotenv` | Leer la clave de Groq del archivo `.env` (en tu PC) |

`gradio`, `pandas`, `uvicorn` y `httpx` ya vienen en Colab: la celda solo los instala si faltan. En tu PC, si ya instalaste `requirements.txt`, no descarga nada.

> Si aparecen avisos rojos de dependencias de paquetes que no usamos (`google-adk`, etc.), puedes ignorarlos.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** instala las librerías que vamos a usar (solo las que falten).
> - **Piénsalo como:** descargar las apps antes de usarlas en el celular.
> - **Qué vas a ver al ejecutarla:** barras de progreso y, quizás, avisos rojos de otros paquetes que puedes ignorar.

In [1]:
# Instalamos las librerías (herramientas ya programadas por otros) que usaremos hoy:
#   langchain + langgraph   → para crear agentes
#   langchain-groq          → para usar el LLM de Groq
#   mcp                     → para crear el servidor MCP y hablarle
#   langchain-mcp-adapters  → para que un agente de LangChain use herramientas MCP
#   a2a-sdk                 → para publicar un agente A2A y hablarle
#   python-dotenv           → para leer la clave de Groq del archivo .env (en tu PC)
# gradio, pandas y uvicorn ya vienen en Colab: solo se instalan si faltan (por ejemplo en tu PC),
# así no cambiamos las versiones de Colab
import importlib.util
MODULOS = {'gradio': 'gradio', 'pandas': 'pandas', 'uvicorn': 'uvicorn'}
faltantes = ' '.join(p for p, m in MODULOS.items() if importlib.util.find_spec(m) is None)
%pip install -q langchain langgraph langchain-groq mcp langchain-mcp-adapters "a2a-sdk[http-server]" python-dotenv {faltantes}

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


> **🧩 En palabras simples**
> - **Qué hace esta celda:** conecta con el LLM de Groq usando tu clave guardada en `.env` (en tu PC) o en Secrets (en Colab).
> - **Piénsalo como:** enchufar el "cerebro" que van a usar todos los agentes.
> - **Qué vas a ver al ejecutarla:** la palabra `conectado`.

In [2]:
import os                              # herramientas para leer variables del sistema
from dotenv import load_dotenv         # lee el archivo .env (en tu PC)
from langchain_groq import ChatGroq    # la "conexión" con el LLM de Groq

# 1) Buscamos la clave de Groq SIN escribirla en el código
load_dotenv()                                    # en tu PC: carga el archivo .env si existe
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    try:
        # En Colab: la leemos de Secrets (icono 🔑 a la izquierda)
        from google.colab import userdata
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except ImportError:
        pass                                     # fuera de Colab este módulo no existe
if not GROQ_API_KEY:
    raise ValueError('Configura GROQ_API_KEY en .env (en tu PC) o en Secrets de Colab.')

# 2) El modelo que usarán TODOS los agentes de hoy
MODELO = 'openai/gpt-oss-120b'

# 3) Una "receta" que entrega un LLM listo para usar.
#    Cada agente pide el suyo (cada uno corre en un lugar distinto y necesita su propia conexión).
def nuevo_llm():
    return ChatGroq(
        model=MODELO,
        api_key=GROQ_API_KEY,
        temperature=0,             # 0 = respuestas lo más estables posible (menos "creatividad")
        reasoning_effort='low',    # que "piense" poco: respuestas más rápidas
    )

# 4) Prueba: le pedimos al LLM que diga "conectado"
print(nuevo_llm().invoke('Responde solo: conectado').content)

conectado


## Paso 1 · ▶️ Bitácora compartida
Cada vez que alguien le envía algo a otro, lo **anotamos** aquí con `registrar(...)`. Gradio leerá esta lista para dibujar el diagrama.

| Campo | Ejemplo |
|---|---|
| `de` → `para` | `Asesor` → `Servidor MCP` |
| `protocolo` | `MCP`, `A2A`, `LLM` o `Local` |
| `accion` | `tools/call consultar_saldo` |
| `detalle` | los datos que viajaron |

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea un cuaderno de registro donde anotaremos cada mensaje entre participantes.
> - **Piénsalo como:** el libro de minutas de una reunión: quién le dijo qué a quién.
> - **Qué vas a ver al ejecutarla:** `Bitácora lista.`

In [3]:
import time, json, logging

# Los servidores escriben muchos mensajes técnicos; los silenciamos para ver solo lo importante
for nombre in ['httpx', 'mcp', 'a2a', 'uvicorn']:
    logging.getLogger(nombre).setLevel(logging.WARNING)

# En Windows, al cerrar cada conexión HTTP, asyncio anota un error inofensivo
# (ConnectionResetError, WinError 10054). Ocultamos SOLO ese aviso; los demás errores se siguen viendo.
logging.getLogger('asyncio').addFilter(lambda registro: '_call_connection_lost' not in registro.getMessage())

# La BITÁCORA es una lista donde anotamos cada mensaje entre participantes.
# Gradio la leerá al final para dibujar las flechas.
BITACORA = []

# Receta para anotar un mensaje en la bitácora:
#   de / para  → quién le habla a quién (ej. 'Asesor' → 'Servidor MCP')
#   protocolo  → por dónde viajó: 'MCP', 'A2A', 'LLM' o 'Local'
#   accion     → qué pidió (ej. 'tools/call consultar_saldo')
#   detalle    → los datos que viajaron
def registrar(de, para, protocolo, accion, detalle=''):
    BITACORA.append({
        'hora': time.strftime('%H:%M:%S'),            # hora del mensaje
        'de': de, 'para': para, 'protocolo': protocolo,
        'accion': accion,
        # si el detalle no es texto (ej. un diccionario), lo convertimos a texto JSON
        'detalle': detalle if isinstance(detalle, str) else json.dumps(detalle, ensure_ascii=False),
    })

print('Bitácora lista.')

Bitácora lista.


# Parte A · MCP: herramientas para cualquier agente

## Paso 2 · ▶️ Crear el servidor MCP
`FastMCP` convierte funciones normales de Python en **herramientas MCP** con un decorador:
- El **nombre** de la función → nombre de la herramienta.
- El **docstring** → descripción (el LLM la lee para decidir cuándo usarla).
- Los **tipos** de los parámetros → esquema JSON de entrada.

El servidor **no sabe nada de LangChain ni de Groq**: cualquier cliente MCP (Claude Desktop, VS Code, otro framework) podría usarlo.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el servidor MCP con dos herramientas: consultar saldo y listar movimientos.
> - **Piénsalo como:** una caja de herramientas con etiquetas claras: cualquier agente que llegue puede leer las etiquetas y usarlas.
> - **Qué vas a ver al ejecutarla:** `Servidor MCP definido con 2 herramientas.`

In [4]:
from mcp.server.fastmcp import FastMCP   # la librería que convierte funciones en herramientas MCP

# ── Los datos del "core bancario" (inventados) ─────────────────────
# Un diccionario: para el cliente DEMO-001 guardamos su saldo y sus movimientos.
CUENTAS = {
    'DEMO-001': {
        'saldo_cop': 5_000_000,
        'movimientos': [
            {'concepto': 'Saldo inicial',    'valor_cop':  4_000_000},
            {'concepto': 'Ingreso ficticio', 'valor_cop':  3_000_000},
            {'concepto': 'Gastos ficticios', 'valor_cop': -2_000_000},
        ],
    }
}

# ── 1) Creamos el servidor MCP y le ponemos nombre ──────────────────
servidor_mcp = FastMCP('Core bancario ficticio', log_level='WARNING')

# ── 2) Primera herramienta: consultar el saldo ──────────────────────
# El decorador @servidor_mcp.tool() convierte la función en una HERRAMIENTA MCP.
# Lo que FastMCP publica para cualquier agente:
#   - el NOMBRE de la función      → nombre de la herramienta ('consultar_saldo')
#   - el texto entre ''' '''       → la descripción que el LLM lee para decidir si la usa
#   - 'cliente_id: str'            → qué dato necesita (un texto)
@servidor_mcp.tool()
def consultar_saldo(cliente_id: str) -> dict:
    '''Devuelve el saldo disponible (COP) de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.'''
    registrar('Asesor', 'Servidor MCP', 'MCP', 'tools/call consultar_saldo', {'cliente_id': cliente_id})   # anotamos el pedido
    cuenta = CUENTAS.get(cliente_id)                                   # buscamos la cuenta
    if cuenta:
        resultado = {'cliente_id': cliente_id, 'saldo_cop': cuenta['saldo_cop']}
    else:
        resultado = {'error': 'cliente no existe'}
    registrar('Servidor MCP', 'Asesor', 'MCP', 'resultado consultar_saldo', resultado)               # anotamos la respuesta
    return resultado                                                   # esto es lo que recibe el agente

# ── 3) Segunda herramienta: listar movimientos ──────────────────────
@servidor_mcp.tool()
def listar_movimientos(cliente_id: str) -> dict:
    '''Devuelve los movimientos y el saldo final de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.'''
    registrar('Asesor', 'Servidor MCP', 'MCP', 'tools/call listar_movimientos', {'cliente_id': cliente_id})
    cuenta = CUENTAS.get(cliente_id)
    if not cuenta:
        resultado = {'error': 'cliente no existe'}
    else:
        movs = cuenta['movimientos']
        # sumamos todos los valores: 4.000.000 + 3.000.000 − 2.000.000 = 5.000.000
        resultado = {'movimientos': movs, 'saldo_final_cop': sum(m['valor_cop'] for m in movs)}
    registrar('Servidor MCP', 'Asesor', 'MCP', 'resultado listar_movimientos', resultado)
    return resultado

print('Servidor MCP definido con 2 herramientas.')

Servidor MCP definido con 2 herramientas.


## Paso 3 · ▶️ Arrancar los servidores en segundo plano
En producción, el servidor MCP y el agente A2A vivirían en **otras máquinas**. Aquí (en Colab o en tu PC) los corremos en **hilos** dentro del mismo kernel, cada uno en su **puerto**:

| Servicio | Dirección |
|---|---|
| Servidor MCP | `http://127.0.0.1:8001/mcp` |
| Agente de Créditos (A2A) | `http://127.0.0.1:9999/` (Paso 7) |

`arrancar(app, puerto)` usa **uvicorn** (un servidor web) en un hilo. Si el puerto ya está ocupado (re-ejecutaste la celda), no hace nada.

> 🔁 MCP también funciona por **stdio** (el cliente lanza el servidor como proceso). Aquí usamos **HTTP** (*streamable HTTP*) porque así el servidor y Gradio comparten la bitácora.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** pone a funcionar el servidor MCP en segundo plano, en la "ventanilla" 8001.
> - **Piénsalo como:** abrir una ventanilla de atención que se queda esperando pedidos.
> - **Qué vas a ver al ejecutarla:** `Servidor escuchando en el puerto 8001`.

In [5]:
import threading, socket, uvicorn   # threading: correr algo "en paralelo"; uvicorn: servidor web

# ¿Ya hay alguien atendiendo en esa ventanilla (puerto)?
def puerto_ocupado(puerto):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', puerto)) == 0   # 0 = sí respondió alguien

# Abre una "ventanilla" (servidor) en el puerto indicado, en segundo plano
def arrancar(app, puerto):
    if puerto_ocupado(puerto):
        # Si ya estaba abierta (por ejemplo, ejecutaste la celda dos veces), no hacemos nada
        print(f'Ya hay un servidor en el puerto {puerto} (reinicia el entorno si cambiaste su código).')
        return
    servidor = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=puerto, log_level='warning'))
    # Lo ponemos a correr "aparte" para que el notebook siga libre
    threading.Thread(target=servidor.run, daemon=True).start()
    while not puerto_ocupado(puerto):    # esperamos hasta que la ventanilla esté abierta
        time.sleep(0.2)
    print(f'Servidor escuchando en el puerto {puerto}')

# La dirección donde vivirá el servidor MCP (127.0.0.1 = "este mismo computador")
URL_MCP = 'http://127.0.0.1:8001/mcp'

# Abrimos la ventanilla del servidor MCP en el puerto 8001
arrancar(servidor_mcp.streamable_http_app(), 8001)

Servidor escuchando en el puerto 8001


## Paso 4 · ▶️ Hablarle a MCP "a mano"
Antes de usar LangChain, veamos **el protocolo desnudo** con el cliente oficial `mcp`:

| Paso del protocolo | Método | Qué devuelve |
|---|---|---|
| 1. Saludo | `initialize` | Nombre y capacidades del servidor |
| 2. Descubrir | `tools/list` | Herramientas con **descripción** y **esquema JSON** de entrada |
| 3. Usar | `tools/call` | El resultado de la herramienta |

**📤 Salida esperada:** el nombre del servidor, 2 herramientas con su esquema y `saldo_cop: 5000000`.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** le habla al servidor MCP paso a paso, sin agente: saludo, pedir el catálogo y usar una herramienta.
> - **Piénsalo como:** llegar a una ventanilla: "hola" → "¿qué servicios tienen?" → "quiero consultar mi saldo".
> - **Qué vas a ver al ejecutarla:** el nombre del servidor, la lista de 2 herramientas con su descripción y el saldo `5000000`.

In [6]:
from mcp import ClientSession                                   # el "cliente" oficial de MCP
from mcp.client.streamable_http import streamable_http_client  # conexión por HTTP

# Abrimos la conexión con el servidor MCP ('async with' la cierra sola al terminar)
async with streamable_http_client(URL_MCP) as (leer, escribir, _):
    async with ClientSession(leer, escribir) as sesion:

        # PASO 1 · initialize = "Hola, ¿quién eres?"
        info = await sesion.initialize()
        print('1) Servidor:', info.serverInfo.name)

        # PASO 2 · tools/list = "¿Qué herramientas ofreces?"
        lista = await sesion.list_tools()
        print('\n2) Herramientas disponibles:')
        for h in lista.tools:                                   # recorremos cada herramienta
            print(f'   • {h.name}: {h.description}')            # su nombre y descripción
            print(f'     entrada: {h.inputSchema["properties"]}')  # qué datos necesita

        # PASO 3 · tools/call = "Usa consultar_saldo con cliente_id = DEMO-001"
        r = await sesion.call_tool('consultar_saldo', {'cliente_id': 'DEMO-001'})
        print('\n3) Resultado de tools/call:', r.content[0].text)

1) Servidor: Core bancario ficticio

2) Herramientas disponibles:
   • consultar_saldo: Devuelve el saldo disponible (COP) de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.
     entrada: {'cliente_id': {'title': 'Cliente Id', 'type': 'string'}}
   • listar_movimientos: Devuelve los movimientos y el saldo final de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.
     entrada: {'cliente_id': {'title': 'Cliente Id', 'type': 'string'}}

3) Resultado de tools/call: {
  "cliente_id": "DEMO-001",
  "saldo_cop": 5000000
}


**🤔 Observa:** nadie le dijo al cliente qué herramientas existían: las **descubrió** con `tools/list`. Eso es lo que permite que **cualquier** agente use **cualquier** servidor MCP.

## Paso 5 · ▶️ De MCP a LangChain: un agente que usa las herramientas
`MultiServerMCPClient` hace los pasos 1–3 por nosotros y convierte cada herramienta MCP en una **herramienta de LangChain**. Luego `create_agent` arma un agente (un grafo de **LangGraph** por dentro):

```
START → modelo (LLM) ─┬─ ¿quiere usar herramienta? → herramientas (MCP) ─┐
                      │                                                  │
                      │ ◄────────────────────────────────────────────────┘
                      └─ no → END
```
**📤 Salida esperada:** el LLM pide `consultar_saldo`, recibe 5.000.000 y responde.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea un agente que usa las herramientas MCP por su cuenta, sin que le digamos cuál usar.
> - **Piénsalo como:** un asesor nuevo que llega, lee las etiquetas de la caja de herramientas y decide solo cuál necesita.
> - **Qué vas a ver al ejecutarla:** la conversación interna: la pregunta, el LLM pidiendo `consultar_saldo`, la herramienta respondiendo y la respuesta final.

In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient   # el "traductor" MCP → LangChain
from langchain.agents import create_agent                        # receta para crear un agente
from langchain.agents.middleware import wrap_tool_call           # para ajustar resultados de herramientas

# Ajuste técnico: MCP devuelve el resultado en "bloques" y Groq necesita texto simple.
# Este "middleware" se pone entre la herramienta y el LLM y convierte los bloques a texto.
@wrap_tool_call
async def resultado_como_texto(request, handler):
    mensaje = await handler(request)      # 1) ejecuta la herramienta normalmente
    mensaje.content = mensaje.text        # 2) convierte el resultado a texto simple
    return mensaje                        # 3) se lo entrega al LLM

# 1) Le decimos al traductor DÓNDE está el servidor MCP y CÓMO hablarle (por HTTP)
cliente_mcp = MultiServerMCPClient({
    'core_bancario': {'transport': 'streamable_http', 'url': URL_MCP},
})

# 2) El traductor hace initialize + tools/list por nosotros y nos entrega las herramientas
herramientas_mcp = await cliente_mcp.get_tools()
print('Herramientas MCP como LangChain:', [h.name for h in herramientas_mcp])

# 3) Creamos un agente: un LLM + instrucciones + las herramientas que puede usar
agente_cuentas = create_agent(
    model=nuevo_llm(),                         # el "cerebro"
    tools=herramientas_mcp,                    # sus "manos": las herramientas MCP
    middleware=[resultado_como_texto],         # el ajuste técnico de arriba
    system_prompt='Eres un asesor de un banco ficticio. El cliente es DEMO-001. Usa las herramientas; no inventes cifras.',
)

# 4) Le hacemos una pregunta (await = esperamos su respuesta)
resultado = await agente_cuentas.ainvoke({'messages': [('user', '¿Cuánto saldo tengo?')]})

# 5) Mostramos la "conversación interna" del agente, mensaje por mensaje:
#    👤 la pregunta → 🤖 el LLM pide una herramienta → 🔧 la herramienta responde → 🤖 respuesta final
for m in resultado['messages']:
    if m.type == 'ai' and m.tool_calls:
        print('🤖 IA pide herramienta:', [(t['name'], t['args']) for t in m.tool_calls])
    elif m.type == 'tool':
        print('🔧 Herramienta responde:', m.text[:120])
    else:
        print(f'{"👤" if m.type == "human" else "🤖"} {m.type}:', m.text[:200])

Herramientas MCP como LangChain: ['consultar_saldo', 'listar_movimientos']
👤 human: ¿Cuánto saldo tengo?
🤖 IA pide herramienta: [('consultar_saldo', {'cliente_id': 'DEMO-001'})]
🔧 Herramienta responde: {
  "cliente_id": "DEMO-001",
  "saldo_cop": 5000000
}
🤖 ai: Estimado cliente **DEMO-001**, su saldo disponible es de **5 000 000 COP**.


# Parte B · A2A: un agente le pide trabajo a otro agente

## Paso 6 · ▶️ Crear el Agente de Créditos
Es un agente **independiente**, con su propio LLM y su propia herramienta **local** (`simular_cuota`, Python puro). Imagina que lo construyó **otro equipo**, quizás con otro framework: al Asesor no le importa, solo necesita hablarle por A2A.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el Agente de Créditos: otro agente, con su propio LLM y su propia calculadora.
> - **Piénsalo como:** un colega experto de otra área, con sus propias herramientas que nadie más ve.
> - **Qué vas a ver al ejecutarla:** `Agente de Créditos definido.`

In [8]:
from langchain.tools import tool   # decorador para convertir una función en herramienta LOCAL

TASA_MENSUAL = 0.01   # 1 % mensual (inventada para la clase)

# Herramienta LOCAL: solo la usa el Agente de Créditos. Nadie de afuera la ve.
@tool
def simular_cuota(precio_cop: int, aporte_cop: int, meses: int) -> dict:
    '''Simula la cuota mensual fija de un crédito ficticio al 1 % mensual.'''
    registrar('Agente Créditos', 'Agente Créditos', 'Local', 'herramienta simular_cuota',
              {'precio': precio_cop, 'aporte': aporte_cop, 'meses': meses})
    capital = precio_cop - aporte_cop                                       # lo que hay que financiar
    cuota = capital * TASA_MENSUAL / (1 - (1 + TASA_MENSUAL) ** (-meses))   # fórmula de cuota fija
    return {'capital_cop': capital, 'meses': meses, 'cuota_aproximada_cop': round(cuota)}

# Receta para crear el Agente de Créditos: su propio LLM + su propia herramienta + sus instrucciones
def crear_agente_creditos():
    return create_agent(
        model=nuevo_llm(),
        tools=[simular_cuota],
        system_prompt='Eres el Agente de Créditos de un banco ficticio. Usa simular_cuota para cualquier cálculo. '
                      'Responde en máximo 3 frases, con las cifras de la herramienta. Nunca apruebes créditos.',
    )

print('Agente de Créditos definido.')

Agente de Créditos definido.


## Paso 7 · ▶️ Publicarlo como servidor A2A
Para que otros agentes lo encuentren y le hablen, A2A pide **tres cosas**:

| Pieza | Para qué | En el código |
|---|---|---|
| **Agent Card** | Su "tarjeta de presentación": nombre, qué sabe hacer (*skills*), dónde escucha | `AgentCard(...)` |
| **Ejecutor** | Qué hacer cuando llega un mensaje: aquí, llamar a nuestro agente de LangGraph | `class EjecutorCreditos(AgentExecutor)` |
| **Rutas HTTP** | `GET /.well-known/agent-card.json` (tarjeta) y `POST /` (mensajes JSON-RPC) | `create_agent_card_routes` + `create_jsonrpc_routes` |

> **🧩 En palabras simples**
> - **Qué hace esta celda:** publica al Agente de Créditos para que otros agentes puedan encontrarlo y hablarle por A2A.
> - **Piénsalo como:** darle al colega una tarjeta de presentación pegada en su puerta y un teléfono (puerto 9999) para que lo llamen.
> - **Qué vas a ver al ejecutarla:** `Servidor escuchando en el puerto 9999`.

In [9]:
from starlette.applications import Starlette                      # para crear una pequeña aplicación web
from a2a.server.agent_execution import AgentExecutor              # molde del "ejecutor"
from a2a.server.request_handlers import DefaultRequestHandler     # quien recibe los mensajes A2A
from a2a.server.tasks import InMemoryTaskStore                    # donde se guardan las tareas
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes   # las "direcciones" web
from a2a.types import AgentCard, AgentInterface, AgentSkill, AgentCapabilities  # piezas de la tarjeta
from a2a.helpers.proto_helpers import new_text_message            # para armar un mensaje de respuesta

# La dirección donde vivirá el Agente de Créditos
URL_A2A = 'http://127.0.0.1:9999/'

# ── PIEZA 1 · AGENT CARD: la "tarjeta de presentación" pública del agente ──
tarjeta = AgentCard(
    name='Agente de Créditos',                                          # quién soy
    description='Simula cuotas de créditos ficticios. No aprueba créditos.',  # qué hago (y qué no)
    version='1.0.0',
    # dónde me encuentran y en qué "idioma" técnico hablo (JSON-RPC, versión 1.0 de A2A)
    supported_interfaces=[AgentInterface(url=URL_A2A, protocol_binding='JSONRPC', protocol_version='1.0')],
    capabilities=AgentCapabilities(streaming=False),   # respondo de una vez, no por partes
    default_input_modes=['text/plain'],                # recibo texto
    default_output_modes=['text/plain'],               # respondo texto
    skills=[AgentSkill(                                # mis habilidades (puede haber varias)
        id='simular_cuota',
        name='Simular cuota',
        description='Calcula la cuota mensual dado precio, aporte y plazo.',
        tags=['crédito', 'simulación'],
        examples=['Simula una moto de 20 millones con 5 de aporte a 36 meses'],
    )],
)

# ── PIEZA 2 · EJECUTOR: qué hacer cuando llega un mensaje ──────────────
# Es el puente entre el "teléfono" A2A y nuestro agente de LangChain.
class EjecutorCreditos(AgentExecutor):
    async def execute(self, context, event_queue):
        pregunta = context.get_user_input()                        # 1) leemos el texto que llegó
        agente = crear_agente_creditos()                           # 2) despertamos al agente
        r = await agente.ainvoke({'messages': [('user', pregunta)]})   # 3) el agente trabaja
        respuesta = r['messages'][-1].text                         # 4) tomamos su respuesta final
        await event_queue.enqueue_event(new_text_message(respuesta))   # 5) la devolvemos por A2A

    async def cancel(self, context, event_queue):
        raise Exception('Cancelar no está soportado en este ejemplo')

# ── PIEZA 3 · RUTAS: las "puertas" web del agente ──────────────────────
#   GET  /.well-known/agent-card.json → muestra la tarjeta
#   POST /                            → recibe mensajes
manejador = DefaultRequestHandler(agent_executor=EjecutorCreditos(),
                                  task_store=InMemoryTaskStore(), agent_card=tarjeta)
app_a2a = Starlette(routes=create_agent_card_routes(tarjeta) + create_jsonrpc_routes(manejador, rpc_url='/'))

# Abrimos la ventanilla del Agente de Créditos en el puerto 9999
arrancar(app_a2a, 9999)

Servidor escuchando en el puerto 9999


## Paso 8 · ▶️ Descubrir al agente y hablarle "a mano"
**8.1 Descubrir:** cualquier cliente pide la Agent Card en una dirección **conocida**: `/.well-known/agent-card.json`.

**8.2 Enviar un mensaje:** un `POST` con **JSON-RPC**, método `SendMessage`. El mensaje tiene `role` y una lista de `parts` (texto, datos, archivos…).

**📤 Salida esperada:** la tarjeta en JSON y luego una respuesta con `"role": "ROLE_AGENT"` y la cuota (≈ $498.215).

> **🧩 En palabras simples**
> - **Qué hace esta celda:** lee la tarjeta del agente y le envía un mensaje "a mano", para ver qué viaja por dentro.
> - **Piénsalo como:** leer la tarjeta en la puerta y luego pasar una nota escrita por debajo.
> - **Qué vas a ver al ejecutarla:** la tarjeta en formato JSON, el mensaje que enviamos y la respuesta del agente con la cuota (≈ $498.215).

In [10]:
import httpx, uuid   # httpx: para hacer pedidos web; uuid: para crear identificadores únicos

# ── 8.1 DESCUBRIR: leer la tarjeta de presentación (como mirar la tarjeta en la puerta) ──
card = httpx.get(URL_A2A + '.well-known/agent-card.json').json()
print('📇 Agent Card:')
print(json.dumps({k: card[k] for k in ['name', 'description', 'skills', 'supportedInterfaces']},
                 indent=2, ensure_ascii=False))

# ── 8.2 ENVIAR: así se ve, por dentro, un mensaje A2A ──
peticion = {
    'jsonrpc': '2.0',                 # el formato del mensaje
    'id': str(uuid.uuid4()),          # número único del pedido (para emparejar la respuesta)
    'method': 'SendMessage',          # QUÉ queremos: enviar un mensaje
    'params': {'message': {
        'messageId': str(uuid.uuid4()),
        'role': 'ROLE_USER',          # quién habla: el "usuario" (aquí, nosotros)
        'parts': [{'text': 'Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses'}],   # el contenido
    }},
}
print('\n📤 Enviamos:', json.dumps(peticion, ensure_ascii=False))

# Lo enviamos por correo web (POST) y esperamos la respuesta
r = httpx.post(URL_A2A, json=peticion, headers={'A2A-Version': '1.0'}, timeout=120)
print('\n📥 Recibimos:', json.dumps(r.json(), indent=2, ensure_ascii=False))

📇 Agent Card:
{
  "name": "Agente de Créditos",
  "description": "Simula cuotas de créditos ficticios. No aprueba créditos.",
  "skills": [
    {
      "id": "simular_cuota",
      "name": "Simular cuota",
      "description": "Calcula la cuota mensual dado precio, aporte y plazo.",
      "tags": [
        "crédito",
        "simulación"
      ],
      "examples": [
        "Simula una moto de 20 millones con 5 de aporte a 36 meses"
      ]
    }
  ],
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:9999/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ]
}

📤 Enviamos: {"jsonrpc": "2.0", "id": "f1e8661b-262b-41c2-b31e-ea84c53cb3ed", "method": "SendMessage", "params": {"message": {"messageId": "a892b344-184e-41e0-abdb-56907ca0d898", "role": "ROLE_USER", "parts": [{"text": "Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses"}]}}}

📥 Recibimos: {
  "result": {
    "message": {
      "messageId": "ac3cd6a7-94be-403d-989f-61e424204b2

## Paso 9 · ▶️ La llamada A2A como herramienta del Asesor
En la práctica no armamos el JSON a mano: usamos el **cliente del SDK** (`create_client`), que lee la Agent Card y elige cómo hablar.

Luego lo envolvemos con `@tool`: para el LLM del Asesor, **pedirle algo a otro agente es una herramienta más**.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** convierte la llamada al Agente de Créditos en una herramienta que el Asesor puede usar.
> - **Piénsalo como:** agregar el número del colega a los contactos del Asesor: ahora puede llamarlo cuando lo necesite.
> - **Qué vas a ver al ejecutarla:** la respuesta del Agente de Créditos con la cuota.

In [11]:
from a2a.client import create_client, ClientConfig                 # el cliente oficial de A2A
from a2a.types import Message, Part, Role, SendMessageRequest      # piezas de un mensaje A2A

# Receta para "llamar por teléfono" al Agente de Créditos y obtener su respuesta en texto
async def enviar_a2a(texto):
    async with httpx.AsyncClient(timeout=120) as http:
        # 1) El cliente lee la tarjeta del agente y prepara la conexión
        cliente = await create_client(URL_A2A, client_config=ClientConfig(streaming=False, httpx_client=http))
        # 2) Armamos el mensaje: lo dice el "usuario", con una parte de texto
        pedido = SendMessageRequest(message=Message(
            role=Role.ROLE_USER, message_id=str(uuid.uuid4()), parts=[Part(text=texto)]))
        # 3) Lo enviamos y leemos lo que responde el agente
        async for evento in cliente.send_message(pedido):
            if evento.HasField('message'):
                return ' '.join(p.text for p in evento.message.parts)
    return '(sin respuesta)'

# Convertimos esa "llamada" en una HERRAMIENTA para el Asesor.
# Para el LLM del Asesor, pedirle algo a otro agente es una herramienta más.
@tool
async def consultar_agente_creditos(pedido: str) -> str:
    '''Delega al Agente de Créditos (vía A2A) cualquier simulación de crédito o cuota.
    Incluye en el pedido el precio, el aporte y el plazo en meses.'''
    registrar('Asesor', 'Agente Créditos', 'A2A', 'SendMessage', pedido)            # anotamos el envío
    respuesta = await enviar_a2a(pedido)                                            # llamamos
    registrar('Agente Créditos', 'Asesor', 'A2A', 'Message (respuesta)', respuesta)   # anotamos la respuesta
    return respuesta

# Prueba directa de la llamada
print(await enviar_a2a('Simula 20.000.000 con aporte de 5.000.000 a 36 meses'))

La cuota mensual aproximada sería de $498 215 COP durante 36 meses. El monto a financiar sería $15 000 000 COP. Recuerde que este cálculo es solo una simulación y no constituye una aprobación de crédito.


# Parte C · Todo junto

## Paso 10 · ▶️ El Asesor: herramientas MCP + un agente A2A
El Asesor recibe **tres herramientas**. Para su LLM todas se ven igual, pero por debajo viajan por protocolos distintos:

| Herramienta | Protocolo | ¿Quién responde? |
|---|---|---|
| `consultar_saldo` | MCP | Servidor MCP (función) |
| `listar_movimientos` | MCP | Servidor MCP (función) |
| `consultar_agente_creditos` | A2A | Agente de Créditos (**otro agente con su propio LLM**) |

**📤 Salida esperada (pregunta mixta):** en la bitácora aparecen eventos **MCP** (saldo) y **A2A** (simulación), y dentro del Agente de Créditos su herramienta **local**.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea al Asesor con tres herramientas (2 MCP + 1 A2A) y le hace una pregunta que necesita ambas.
> - **Piénsalo como:** el asesor de la sucursal: consulta el sistema (MCP) y llama al colega de créditos (A2A).
> - **Qué vas a ver al ejecutarla:** una tabla con cada mensaje: quién habló con quién y por qué protocolo; y la respuesta final.

In [12]:
import pandas as pd   # para mostrar la bitácora como tabla

# Receta para crear al ASESOR: el agente principal que habla con el cliente
async def crear_asesor():
    # Sus herramientas: las 2 del servidor MCP + la llamada al agente A2A
    herramientas = await cliente_mcp.get_tools() + [consultar_agente_creditos]
    return create_agent(
        model=nuevo_llm(),
        tools=herramientas,
        middleware=[resultado_como_texto],
        system_prompt=('Eres el Asesor de un banco ficticio. El cliente es DEMO-001. '
                       'Para saldo o movimientos usa las herramientas MCP. '
                       'Para simular créditos o cuotas delega con consultar_agente_creditos. '
                       'No inventes cifras. Responde en español, breve. No apruebes créditos.'),
    )

# Una pregunta "mixta": necesita MCP (saldo) y A2A (simulación)
PREGUNTA = ('¿Cuánto saldo tengo? Y simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses')

BITACORA.clear()                                                    # borramos anotaciones anteriores
registrar('Usuario', 'Asesor', 'LLM', 'pregunta', PREGUNTA)         # anotamos la pregunta
asesor = await crear_asesor()                                       # creamos al Asesor
final = await asesor.ainvoke({'messages': [('user', PREGUNTA)]})    # el Asesor trabaja
respuesta = final['messages'][-1].text                              # su respuesta final
registrar('Asesor', 'Usuario', 'LLM', 'respuesta', respuesta)

# Mostramos la bitácora: cada fila es un mensaje entre participantes
with pd.option_context('display.max_colwidth', 90):
    display(pd.DataFrame(BITACORA)[['de', 'para', 'protocolo', 'accion', 'detalle']])
print('\n💬', respuesta)

,de,para,protocolo,accion,detalle
0,Usuario,Asesor,LLM,pregunta,¿Cuánto saldo tengo? Y simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses
1,Asesor,Servidor MCP,MCP,tools/call consultar_saldo,"{""cliente_id"": ""DEMO-001""}"
2,Servidor MCP,Asesor,MCP,resultado consultar_saldo,"{""cliente_id"": ""DEMO-001"", ""saldo_cop"": 5000000}"
3,Asesor,Agente Créditos,A2A,SendMessage,Simular crédito moto de 20000000 con aporte de 5000000 a 36 meses
4,Agente Créditos,Agente Créditos,Local,herramienta simular_cuota,"{""precio"": 20000000, ""aporte"": 5000000, ""meses"": 36}"
5,Agente Créditos,Asesor,A2A,Message (respuesta),La cuota mensual sería aproximadamente **$498.215** COP por 36 meses. El monto financi...
6,Asesor,Usuario,LLM,respuesta,Su saldo disponible es **$5.000.000 COP**.\n\nLa simulación para la moto de $20.000.00...



💬 Su saldo disponible es **$5.000.000 COP**.

La simulación para la moto de $20.000.000 con aporte de $5.000.000 a 36 meses indica una cuota mensual aproximada de **$498.215 COP** (financiando $15.000.000). 

Recuerde que esta es solo una simulación y no constituye una aprobación de crédito.


## Paso 11 · ▶️ Gradio: diagrama de secuencia en vivo
Cada columna es un participante; cada fila, un mensaje. El color dice **por qué protocolo** viajó:

| Color | Protocolo |
|---|---|
| 🟦 azul | Usuario ↔ Asesor |
| 🟩 verde | **MCP** (herramientas) |
| 🟪 morado | **A2A** (agente ↔ agente) |
| 🟧 naranja | Herramienta **local** del agente remoto |

Mientras el Asesor trabaja, la función `atender` revisa la bitácora cada 0,4 s y **redibuja** (`yield`).

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea la interfaz visual con el diagrama de flechas.
> - **Piénsalo como:** una pantalla que dibuja en vivo quién le habla a quién.
> - **Qué vas a ver al ejecutarla:** la interfaz (en tu PC, en `http://127.0.0.1:7860`; en Colab, además, un enlace público): ábrela y escribe una pregunta.

In [13]:
# ═════════════════════════════════════════════════════════════════════
# CÓMO LEER ESTA CELDA (no necesitas entender el HTML):
#  1) CARRILES / ICONOS / COLORES → las columnas del diagrama y el color de cada protocolo
#  2) dibujar_secuencia(...)      → convierte la bitácora en flechas de colores (HTML)
#  3) tabla(...)                  → convierte la bitácora en filas para la tabla
#  4) atender(pregunta)           → lo que pasa al hacer clic en "Enviar":
#        - borra la bitácora, crea al Asesor y lo pone a trabajar
#        - mientras trabaja, cada 0,4 segundos redibuja las flechas (por eso se ven "en vivo")
#  5) with gr.Blocks(...)         → arma la pantalla: caja de texto, ejemplos, botón, diagrama y tabla
#  6) demo.launch(...)            → abre la interfaz (enlace público solo en Colab)
# ═════════════════════════════════════════════════════════════════════
import asyncio
import gradio as gr

CARRILES = ['Usuario', 'Asesor', 'Servidor MCP', 'Agente Créditos']   # columnas del diagrama
ICONOS   = {'Usuario': '👤', 'Asesor': '🧑‍💼', 'Servidor MCP': '🗄️', 'Agente Créditos': '🤖'}
COLORES  = {'LLM': '#2563eb', 'MCP': '#0d9488', 'A2A': '#7c3aed', 'Local': '#d97706'}

def dibujar_secuencia(eventos):
    # Encabezado: una columna por participante (colores fijos: se lee en tema claro y oscuro)
    cab = ''.join(f'<div style="text-align:center;font-weight:700;color:#0f172a;padding:8px;'
                  f'background:#e2e8f0;border-radius:8px">{ICONOS[c]} {c}</div>' for c in CARRILES)
    filas = ''
    for i, ev in enumerate(eventos, start=1):
        a, b = CARRILES.index(ev['de']), CARRILES.index(ev['para'])
        color = COLORES[ev['protocolo']]
        izq, der = min(a, b), max(a, b)
        etiqueta = f'{i}. [{ev["protocolo"]}] {ev["accion"]}'
        if a == b:   # mensaje a sí mismo (herramienta local)
            flecha = f'<div style="border:2px dashed {color};border-radius:8px;padding:2px;text-align:center;color:{color};font-weight:700">↻ trabaja por dentro</div>'
        else:        # flecha de izquierda a derecha o al revés
            punta = '▶' if b > a else '◀'
            linea = f'<div style="flex:1;height:3px;background:{color}"></div>'
            flecha = (f'<div style="display:flex;align-items:center;color:{color}">'
                      + (linea + punta if b > a else punta + linea) + '</div>')
        filas += (f'<div style="grid-column:{izq + 1}/{der + 2};padding:2px 12%">'
                  f'<div style="font-size:12px;color:{color};font-weight:600;text-align:center">{etiqueta}</div>'
                  f'{flecha}</div>')
    return (f'<div style="background:#ffffff;border-radius:12px;padding:12px;color:#0f172a">'
            f'<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:6px 0">{cab}{filas}</div>'
            f'{"" if eventos else "<p style=color:#64748b>Sin mensajes todavía</p>"}</div>')

def tabla(eventos):
    return [[i, e['de'], e['para'], e['protocolo'], e['accion'], e['detalle'][:160]]
            for i, e in enumerate(eventos, start=1)]

async def atender(pregunta):
    BITACORA.clear()
    registrar('Usuario', 'Asesor', 'LLM', 'pregunta', pregunta)
    yield dibujar_secuencia(BITACORA), tabla(BITACORA), '⏳ Pensando…'

    asesor = await crear_asesor()
    trabajo = asyncio.create_task(asesor.ainvoke({'messages': [('user', pregunta)]}))   # corre en paralelo
    while not trabajo.done():                      # mientras trabaja, redibujamos lo que va pasando
        await asyncio.sleep(0.4)
        yield dibujar_secuencia(BITACORA), tabla(BITACORA), '⏳ Pensando…'

    try:
        respuesta = trabajo.result()['messages'][-1].text
    except Exception as error:
        respuesta = f'⛔ Error: {error}'
    registrar('Asesor', 'Usuario', 'LLM', 'respuesta', respuesta)
    yield dibujar_secuencia(BITACORA), tabla(BITACORA), respuesta

with gr.Blocks(title='MCP y A2A') as demo:
    gr.Markdown('# 🏦 MCP y A2A en acción\n🟩 **MCP** = herramientas · 🟪 **A2A** = otro agente · 🟧 herramienta local del agente remoto. Datos ficticios.')
    pregunta = gr.Textbox(label='Pregunta del cliente', value='¿Cuánto saldo tengo?')
    gr.Examples([['¿Cuánto saldo tengo?'],
                 ['Muéstrame mis movimientos'],
                 ['Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses'],
                 ['¿Cuánto saldo tengo? Y simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses']],
                inputs=pregunta)
    boton = gr.Button('Enviar', variant='primary')
    secuencia = gr.HTML()
    respuesta = gr.Textbox(label='Respuesta del Asesor', lines=4)
    detalle = gr.Dataframe(headers=['#', 'De', 'Para', 'Protocolo', 'Acción', 'Datos'],
                           wrap=True, interactive=False, label='Mensajes (qué viajó)',
                           column_widths=['5%', '12%', '12%', '9%', '20%', '42%'])
    boton.click(atender, pregunta, [secuencia, detalle, respuesta])

# ¿Estamos en Colab? (el módulo google.colab solo existe allí)
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# share=EN_COLAB → en Colab crea además un enlace público temporal (solo datos ficticios);
#                  en tu PC la interfaz solo se abre en tu equipo (http://127.0.0.1:7860)
# debug=False    → la celda termina y puedes seguir ejecutando otras celdas
demo.launch(share=EN_COLAB, debug=False)

c:\Users\A47x\OneDrive\A47x\ARQUITECTURAS-EN-INTELIGENCIA-ARTIFICIAL-MULTIAGENTICA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Paso 12 · ▶️ Cerrar la interfaz

> **🧩 En palabras simples**
> - **Qué hace esta celda:** cierra la interfaz.
> - **Piénsalo como:** cerrar la ventanilla al final del día.
> - **Qué vas a ver al ejecutarla:** nada, o un mensaje de cierre.

In [14]:
# Apaga el servidor de Gradio (libera el puerto y, en Colab, el enlace público)
demo.close()

Closing server running on port: 7860


## 🎯 Recorrido guiado en la interfaz
| # | Pregunta | Qué observar |
|---|---|---|
| 1 | «¿Cuánto saldo tengo?» | Solo flechas **verdes** (MCP): el Asesor usa una herramienta |
| 2 | «Muéstrame mis movimientos» | Otra herramienta del **mismo** servidor MCP |
| 3 | «Simula una moto…» | Flecha **morada** (A2A) al Agente de Créditos y, dentro de él, ↻ **naranja**: su herramienta local. El Asesor **no ve** esa herramienta: solo recibe la respuesta |
| 4 | Pregunta mixta | MCP **y** A2A en la misma conversación. Las flechas pueden **intercalarse**: el LLM puede pedir ambas herramientas **a la vez** y se ejecutan en paralelo |

## 🤔 Preguntas para discutir
1. ¿Qué diferencia hay entre lo que devuelve `consultar_saldo` (MCP) y lo que devuelve el Agente de Créditos (A2A)?
2. Si mañana el Agente de Créditos se reescribe en otro framework, ¿qué hay que cambiar en el Asesor? (Pista: nada, mientras respete A2A.)
3. ¿Cuándo convertirías una **herramienta** en un **agente**? (Pista: cuando necesita razonar, varios pasos o su propio equipo.)
4. ¿Qué riesgos aparecen si un servidor MCP o un agente A2A es de un tercero? (Autenticación, permisos, datos sensibles.)

## 🧪 Ejercicio
1. Agrega al servidor MCP una herramienta `consultar_tasa()` que devuelva la tasa ficticia. **Reinicia el entorno** y re-ejecuta: el Asesor la descubre sola.
2. Crea un **Agente de Requisitos** A2A en el puerto `9998` (copia el paso 7) y conéctalo al Asesor con una nueva herramienta.

## ⚠️ Errores frecuentes
| Síntoma | Causa | Qué hacer |
|---|---|---|
| `Ya hay un servidor en el puerto…` y no ves tus cambios | El servidor viejo sigue corriendo en su hilo | Reinicia el kernel (**Entorno de ejecución → Reiniciar** en Colab, **Restart** en VS Code) y ejecuta todo |
| Error 401 de Groq | Falta `GROQ_API_KEY` o no diste acceso al notebook | En tu PC: revisa `.env`. En Colab: 🔑 Secrets → activar acceso |
| Error 429 | Muchas llamadas (cada agente usa el LLM) | Espera unos segundos |
| El Asesor responde sin usar herramientas | El LLM decidió no usarlas | Haz la pregunta más explícita |

## Fuentes (consultadas el 28-10-2026)
- [Model Context Protocol](https://modelcontextprotocol.io) · [SDK de Python `mcp`](https://pypi.org/project/mcp/)
- [LangChain MCP adapters](https://docs.langchain.com/oss/python/langchain/mcp)
- [Protocolo A2A](https://a2a-protocol.org) · [SDK de Python `a2a-sdk`](https://pypi.org/project/a2a-sdk/)
- [LangChain `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)